<a href="https://colab.research.google.com/github/Akshatha7710/smart-tea-estate-management-system/blob/soil_quality-analysis_and_predictive_modeling/Soil_Quality_Analysis_and_Predictive_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==========================================================
# STEMS - Smart Tea Estate Management System
# Rainfall + Wet Days Forecasting + Soil Prediction
# ==========================================================

# ==========================================================
# 1. Import Required Libraries
# ==========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

from statsmodels.tsa.statespace.sarimax import SARIMAX


# ==========================================================
# 2. Load Datasets
# ==========================================================

climate = pd.read_csv("sample data/climate_data.csv")
soil = pd.read_csv("sample data/soil_data.csv")


# ==========================================================
# 3. Climate Data Preprocessing
# ==========================================================

month_map = {
    "Jan":1,"Feb":2,"Mar":3,"Apr":4,"May":5,"Jun":6,
    "Jul":7,"Aug":8,"Sep":9,"Oct":10,"Nov":11,"Dec":12
}

climate["Month"] = climate["Month"].map(month_map)

# Create date column
climate["Date"] = pd.to_datetime(dict(year=climate.Year,
                                      month=climate.Month,
                                      day=1))

climate = climate.sort_values("Date")


# ==========================================================
# 4. Feature Engineering
# ==========================================================

climate["Rainfall_lag1"] = climate["Rainfall"].shift(1)
climate["Rainfall_lag2"] = climate["Rainfall"].shift(2)

climate["WetDays_lag1"] = climate["WetDays"].shift(1)

climate["Rainfall_3month_avg"] = climate["Rainfall"].rolling(3).mean()

climate = climate.dropna()


# ==========================================================
# 5. Rainfall Forecast Model (SARIMAX)
# ==========================================================

rain_series = climate.set_index("Date")["Rainfall"]

rain_model = SARIMAX(
    rain_series,
    order=(1,1,1),
    seasonal_order=(1,1,1,12)
)

rain_results = rain_model.fit()

# Forecast rainfall
rain_forecast = rain_results.forecast(steps=12)

print("\nRainfall Forecast (Next 12 Months)")
print(rain_forecast)


# ==========================================================
# 6. Wet Days Prediction Model
# ==========================================================

features = [
    "Rainfall",
    "Rainfall_lag1",
    "Rainfall_lag2",
    "Rainfall_3month_avg"
]

X = climate[features]
y = climate["WetDays"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

wet_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=6,
    random_state=42
)

wet_model.fit(X_train, y_train)

wet_pred = wet_model.predict(X_test)

print("\nWet Days Prediction Performance")
print("MAE:", mean_absolute_error(y_test, wet_pred))
print("R2:", r2_score(y_test, wet_pred))


# ==========================================================
# 7. Soil Prediction Model
# ==========================================================

# Create field age
soil["FieldAge"] = 2025 - soil["PlantingYear"]

# Add climate indicators
soil["AvgRainfall"] = climate["Rainfall"].mean()
soil["AvgWetDays"] = climate["WetDays"].mean()

soil_features = [
    "Area",
    "FieldAge",
    "AvgRainfall",
    "AvgWetDays"
]

X_soil = soil[soil_features]
y_soil = soil["pH"]

X_train, X_test, y_train, y_test = train_test_split(
    X_soil, y_soil, test_size=0.2, random_state=42
)

soil_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=7,
    random_state=42
)

soil_model.fit(X_train, y_train)

soil_pred = soil_model.predict(X_test)

print("\nSoil pH Prediction Performance")
print("MAE:", mean_absolute_error(y_test, soil_pred))
print("R2:", r2_score(y_test, soil_pred))


# ==========================================================
# 8. Visualization
# ==========================================================

plt.figure(figsize=(10,5))

plt.plot(rain_series, label="Historical Rainfall")

plt.plot(rain_forecast, label="Forecast Rainfall")

plt.title("Rainfall Forecast")
plt.legend()

plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'sample data/climate_data.csv'